In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
import joblib

In [2]:
data_path = '../data/'
file_to_load = 'Friday-WorkingHours-Morning.pcap_ISCX.csv'

print(f"[*] Đang nạp {file_to_load}...")
df = pd.read_csv(os.path.join(data_path, file_to_load))

print(f"[+] Kích thước dữ liệu ban đầu: {df.shape}")

# Xem thử các nhãn (Labels) có trong file này
print("\nPhân phối các loại tấn công:")
print(df[' Label'].value_counts())

[*] Đang nạp Friday-WorkingHours-Morning.pcap_ISCX.csv...
[+] Kích thước dữ liệu ban đầu: (191033, 79)

Phân phối các loại tấn công:
 Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64


In [3]:
# 1. Xóa khoảng trắng ở tên cột (Ví dụ: ' Label' -> 'Label')
df.columns = df.columns.str.strip()

# 2. Xử lý giá trị rác (NaN và Infinity)
print("[*] Đang làm sạch dữ liệu Infinity và NaN...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 3. Gán nhãn nhị phân (0: Mạng bình thường, 1: Mạng độc hại/C2)
# Trong file Friday Morning, lưu lượng độc hại thường được dán nhãn là 'Bot'
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print(f"[+] Kích thước dữ liệu sau khi làm sạch: {df.shape}")
print("\nPhân phối nhãn mới (0: Sạch, 1: Độc hại):")
print(df['Label'].value_counts())

[*] Đang làm sạch dữ liệu Infinity và NaN...
[+] Kích thước dữ liệu sau khi làm sạch: (190911, 79)

Phân phối nhãn mới (0: Sạch, 1: Độc hại):
Label
0    188955
1      1956
Name: count, dtype: int64


In [4]:
# Chọn các cột liên quan đến kích thước gói tin và thời gian (Inter-Arrival Time)
selected_features = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Length of Fwd Packets',
    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min'
]

# Tách dữ liệu thành X (Features) và y (Labels)
X = df[selected_features]
y = df['Label']

print(f"[+] Đã trích xuất {len(selected_features)} đặc trưng cốt lõi.")
X.head()

[+] Đã trích xuất 11 đặc trưng cốt lõi.


,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min
0,112740690,32,6448,403,0,201.5,204.724205,2.398738e+06,5.798698e+06,16400000,3
1,112740560,32,6448,403,0,201.5,204.724205,2.398735e+06,5.798710e+06,16400000,2
2,113757377,545,0,0,0,0.0,0.000000,2.091128e+05,1.395543e+06,20800000,0
3,100126,22,616,28,28,28.0,0.000000,4.767905e+03,2.183302e+04,100055,1
4,54760,4,0,0,0,0.0,0.000000,1.825333e+04,3.046984e+04,53431,108


In [5]:
from imblearn.over_sampling import SMOTE

print(f"[*] Phân phối nhãn trước khi cân bằng: \n{y.value_counts()}\n")
print("[*] Đang áp dụng thuật toán SMOTE (Có thể mất 1-2 phút)...")

# Khởi tạo SMOTE với random_state để đảm bảo kết quả nhất quán giữa các lần chạy
smote = SMOTE(sampling_strategy='auto', random_state=42)

# Thực hiện sinh dữ liệu
X_balanced, y_balanced = smote.fit_resample(X, y)

print(f"[+] Kích thước ma trận đặc trưng mới: {X_balanced.shape}")
print("[+] Phân phối nhãn sau khi cân bằng: \n", y_balanced.value_counts())

[*] Phân phối nhãn trước khi cân bằng: 
Label
0    188955
1      1956
Name: count, dtype: int64

[*] Đang áp dụng thuật toán SMOTE (Có thể mất 1-2 phút)...
[+] Kích thước ma trận đặc trưng mới: (377910, 11)
[+] Phân phối nhãn sau khi cân bằng: 
 Label
0    188955
1    188955
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_balanced # Đảm bảo tỷ lệ 50-50 được giữ nguyên ở cả 2 tập
)

print(f"[*] Tập huấn luyện (Train): {X_train.shape[0]} mẫu")
print(f"[*] Tập kiểm thử (Test): {X_test.shape[0]} mẫu")

[*] Tập huấn luyện (Train): 302328 mẫu
[*] Tập kiểm thử (Test): 75582 mẫu


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("[*] Đang khởi tạo và huấn luyện Random Forest (The Judge)...")
# Khởi tạo mô hình. Giới hạn độ sâu (max_depth=15) để tránh overfit, 
# đồng thời giữ tốc độ suy luận (predict) trong môi trường thời gian thực đủ nhanh.
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    random_state=42, 
    n_jobs=-1
)

# Đưa ma trận đặc trưng đã chia (từ dữ liệu SMOTE) vào huấn luyện
rf_model.fit(X_train, y_train)

print("[*] Đang đánh giá hiệu năng trên tập Test...")
y_pred = rf_model.predict(X_test)
print("\nBáo cáo phân loại (Classification Report):")
print(classification_report(y_test, y_pred, digits=4))

# Xuất mô hình ra file .pkl vào thư mục data dùng chung
model_path = '../data/ids_model.pkl'
joblib.dump(rf_model, model_path)

# (Tùy chọn) Lưu luôn danh sách tên cột đặc trưng để Actuator nạp vào đúng thứ tự
feature_path = '../data/ids_features.pkl'
joblib.dump(selected_features, feature_path)

print(f"\n[+] Xuất xưởng thành công! File mô hình đã được lưu tại: {model_path}")
print("[+] Blue Team hoàn tất nhiệm vụ. Khối mạng và AI đã có thể nạp file này để chấm điểm.")

[*] Đang khởi tạo và huấn luyện Random Forest (The Judge)...
[*] Đang đánh giá hiệu năng trên tập Test...

Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

           0     0.9990    0.9720    0.9853     37791
           1     0.9727    0.9990    0.9857     37791

    accuracy                         0.9855     75582
   macro avg     0.9858    0.9855    0.9855     75582
weighted avg     0.9858    0.9855    0.9855     75582


[+] Xuất xưởng thành công! File mô hình đã được lưu tại: ../data/ids_model.pkl
[+] Blue Team hoàn tất nhiệm vụ. Khối mạng và AI đã có thể nạp file này để chấm điểm.
